# 00 · Build data (CPU, or Colab GPU for a local teacher)

Prose corpus → non-IID clients (prose shards + log slices + benign) → 4 task datasets → train/val/test splits.

**Set up first:** copy `.env.example`→`.env`. For matched both-class log data with no Splunk lab, download a paper-backed dataset (e.g. AIT), convert it below, and point `FEDDAPT_LOG_SOURCES` at the output.

In [ ]:
# --- setup: run once per Colab session ---
import os
# 1) get the code (skip if you opened this notebook from the GitHub tab)
if not os.path.exists('pyproject.toml'):
    if not os.path.exists('fedapt'):
        get_ipython().system('git clone https://github.com/YOUR_USERNAME/fedapt.git')
    get_ipython().run_line_magic('cd', 'fedapt')
# 2) install (keep the extras — Colab has the GPU for [train]/[eval])
get_ipython().system('pip -q install -e "."')
# 3) persist corpus/adapters/results to Google Drive so a dropped session resumes.
#    On Colab, Config auto-defaults FEDDAPT_ROOT to /content/drive/MyDrive/FedDAPT.
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

### Optional: run a local teacher on Colab's GPU (no API key)
Run this cell to install Ollama + pull a model, then select it as the teacher below. Skip it if you use an API teacher or want offline metadata fallbacks.

In [ ]:
# --- OPTIONAL: run a strong LOCAL teacher/judge on Colab's GPU via Ollama ---
# The open-weight models (Qwen/Gemma/Llama) run on the SAME GPU as training, so
# only do this during data-build (nb 00) or the judge pass (nb 03) — never during
# training. T4: qwen3:14b / gemma3:12b.  A100: qwen3:32b / gemma3:27b.
get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')
import subprocess, time, os
subprocess.Popen(['ollama', 'serve'])          # background server
time.sleep(5)
get_ipython().system('ollama pull qwen3:14b')
os.environ['FEDDAPT_OLLAMA_HOST'] = 'http://localhost:11434'

In [ ]:
from fedapt.config import load_config
from fedapt import corpus, clients, tasks, splits
cfg = load_config()
print('root =', cfg.root)

### Optional: convert a paper-backed dataset (both classes, matched)
Produces a normalized `.jsonl`; set `FEDDAPT_LOG_SOURCES=data/normalized` in `.env`, then rebuild.

In [ ]:
# after downloading AIT to Drive (both benign+attack, one matched environment):
# !python scripts/convert_dataset.py --dataset ait \
#     --src /content/drive/MyDrive/AIT/<dataset> --out data/normalized/ait.jsonl

### Teacher for real task targets (else `teacher=None` = metadata fallbacks)

In [ ]:
teacher = None
from fedapt.judge import make_llm
# LOCAL model on Colab's GPU (run the Ollama cell first):
# teacher = make_llm('ollama:qwen3:14b')
# or an API model (needs ANTHROPIC_API_KEY in .env):
# teacher = make_llm('claude-haiku-4-5')

In [ ]:
corpus.build_corpus(cfg)
clients.build_clients(cfg)
tasks.build_tasks(cfg, teacher=teacher)   # watch the '⚠ fell back' line — it should be 0
splits.build_splits(cfg)

Next → **01 Federated DAPT** (GPU).